# Stage 0 — routing analysis

Interactive version of `ff-analyze`. Collect a trace first:

```bash
uv run ff-collect --out traces/olmoe
```

Six questions, each gating a later design decision:

| | Question | Decides |
|---|---|---|
| **Q1** | How skewed is expert usage? | whether a small pinned cache is viable |
| **Q2** | Does token *t+1* reuse token *t*'s experts? | whether recency-based eviction makes sense |
| **Q3** | Can layer *N* predict layer *N+k*'s routing? | **whether prefetch works at all** |
| **Q4** | Does usage cluster by domain? | whether cache warming pays |
| **Q5** | Hit rate vs capacity, incl. Belady | how much headroom any online policy has |
| **Q6** | How fast does the expert set grow with block size? | whether batching/speculation amortises loads |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from flashforge import analysis, cachesim, plots, viz

viz.use_style()

TRACES = Path.cwd().parent / "traces" / "olmoe"
store = analysis.TraceStore(TRACES)
frame = store.routing

print(f"model        {store.meta['model_id']}")
print(f"experts      {store.num_experts} per layer, top-{store.top_k}")
print(f"MoE layers   {len(store.moe_layers)}")
print(f"slots        {store.num_experts * len(store.moe_layers)} distinct (layer, expert)")
print(f"tokens       {store.meta['total_tokens']:,} across {store.meta['n_sequences']} sequences")
print(f"routing rows {len(frame):,}")

## Q1 — how skewed is expert usage?

If the hottest 10% of experts serve far more than 10% of accesses, a small
pinned cache captures a disproportionate share of traffic and the whole
caching approach has something to work with. If usage is near-uniform,
caching degenerates to "hold as much as fits" and the interesting work moves
entirely to prefetch.

In [ ]:
freq = analysis.expert_frequency(frame, store.num_experts)
skew = analysis.skew_summary(freq, store.num_experts)
lorenz = analysis.lorenz_curve(freq)

print(f"hottest 10% of experts serve {skew['top10pct_mass'].mean():.1%} of accesses "
      f"(uniform would be 10.0%)")
print(f"mean Gini {skew['gini'].mean():.3f} | "
      f"never-routed experts: {int(skew['unused_experts'].sum())}")

display(skew.round(3))
plots.plot_skew(lorenz, skew);

## Q2 — temporal locality

Overlap at lag 1 is the fraction of a token's experts that the previous token
also used. Compare against the random baseline `top_k / num_experts`. High
lag-1 overlap that decays with distance is the signature recency-based
eviction exploits; a flat curve means LRU has no edge over LFU.

In [ ]:
overlap = analysis.consecutive_overlap(frame, store.num_experts, store.top_k)

by_lag = overlap.groupby("lag")["mean_overlap"].mean()
print(f"random baseline: {store.top_k / store.num_experts:.1%}\n")
for lag, value in by_lag.items():
    print(f"  lag {lag:>2}  {value:.1%}")

plots.plot_locality(overlap);

In [ ]:
# Reuse-distance tail: short gaps are what recency catches, the long tail is
# what a frequency-based policy picks up instead.
reuse = analysis.reuse_distance(frame)
display(reuse[["median_gap", "p90_gap", "n_reuses"]].describe().round(2))

## Q3 — cross-layer predictability

**This is the question that decides the project.**

Prefetch only pays if you have lead time. Predicting layer *N+k*'s routing at
layer *N* buys *k* layers of compute to hide the transfer behind. If recall
collapses at *k*=1, there is no window and the design has to change.

Read the results as a ladder:

- `prior` is the floor. A predictor that can't beat a static frequency table is worthless.
- `stale_router` is the one to beat — and often the one to ship. It runs layer *N+k*'s **actual** router on layer *N*'s hidden state: no training, no extra parameters, free at inference time.
- `probe` approximates the ceiling for a learned predictor of this size. If it barely beats `stale_router`, don't bother training anything.

Budget 2x models over-prefetching: fetch `2 * top_k` experts and hope the true
`top_k` are among them. Cheap insurance if bandwidth allows.

In [ ]:
detail = analysis.cross_layer_predictability(store, offsets=(1, 2, 4), max_tokens=20_000)
summary = analysis.predictability_summary(detail)

display(
    summary.pivot(index=["budget_mult", "offset"], columns="predictor", values="recall").round(3)
)
plots.plot_predictability(summary, store.top_k);

In [ ]:
# Does predictability vary by depth? If early layers are predictable and late
# ones are not (or vice versa), the prefetch policy should be depth-dependent
# rather than uniform.
by_depth = (
    detail[(detail.offset == 1) & (detail.budget_mult == 1) & (detail.predictor == "stale_router")]
    .sort_values("src_layer")[["src_layer", "tgt_layer", "recall"]]
)
display(by_depth.round(3).to_string(index=False))

## Q4 — does usage cluster by domain?

If different kinds of text route to distinguishable expert sets, you can warm
the cache per conversation.

The verdict compares between-domain divergence against a **sample-matched**
within-domain noise floor. Matching matters: JS divergence estimated from
fewer tokens is biased upward simply because sparse profiles differ by chance,
so an unmatched baseline can bury a real signal.

In [ ]:
result = analysis.domain_divergence(frame, store.domains, store.num_experts)

print(f"between-domain JS  {result.between_matched:.4f}")
print(f"within-domain floor {result.within_matched:.4f}")
print(f"ratio               {result.ratio:.2f}x")
print()
print("domains separate — warming has something to exploit"
      if result.separates else
      "domains do NOT separate cleanly — warming is unlikely to pay")

display(result.matrix.round(4))
plots.plot_domain(result);

## Q5 — cache simulation

Belady is the ceiling for **demand-paging**: it evicts the entry whose next
use is furthest away, which no online policy can beat without seeing the
future. Read the LRU→Belady gap as your budget for smarter eviction:

- **narrow gap** → eviction is nearly solved; spend the effort on prefetch.
- **wide gap** → a better eviction policy is worth real work.

`static` can score marginally *above* Belady, which is not a contradiction: it
loads at *t*=0 and so skips the compulsory first-touch misses a demand policy
must pay. That small edge is this project's thesis in miniature — prefetching
attacks misses that no eviction rule can reach.

**Watch for the LRU cliff.** MoE decode is a *cyclic* pattern: each token sweeps
every layer, touching roughly `top_k * n_layers` distinct experts before
returning to layer 0. That is LRU's textbook worst case — below a capacity of
one token's working set, LRU evicts every entry just before its next use. In the
synthetic check its hit rate goes to *exactly zero* while LFU and static are
still at 25–40% on the same trace. If you see that cliff here, a single global
LRU is the wrong default, and the fix is per-layer cache partitioning or a
frequency-biased policy. Check where LRU crosses LFU before designing anything.

In [ ]:
# One expert's weights, in bytes. Adjust for your model: OLMoE is
# 3 matrices x hidden x intermediate x 2 bytes (fp16).
BYTES_PER_EXPERT = 3 * 2048 * 1024 * 2  # ~12.6 MB

keys = cachesim.build_access_sequence(frame, store.num_experts)
total_slots = store.num_experts * len(store.moe_layers)
MAX_ACCESSES = 300_000

capacities = sorted({
    max(1, int(total_slots * f))
    for f in (0.01, 0.02, 0.05, 0.10, 0.15, 0.25, 0.40, 0.60, 0.80, 1.00)
})
sweep = cachesim.sweep(
    keys[:MAX_ACCESSES], capacities,
    bytes_per_expert=BYTES_PER_EXPERT,
    accesses_per_token=store.top_k * len(store.moe_layers),
)

display(sweep.pivot(index="capacity", columns="policy", values="hit_rate").round(4))
plots.plot_cache(sweep, total_slots=total_slots);

In [ ]:
# Turn hit rates into wall-clock. This is the number that decides the hardware
# question: fetch volume per token divided by the bandwidth of whichever tier
# the misses come from.
TIERS = {
    "HDD 7200rpm": 1.2e8,
    "SATA SSD": 5.0e8,
    "NVMe Gen3 x4": 3.0e9,
    "NVMe Gen4 x4": 6.5e9,
    "DDR4-2133 dual": 3.0e10,
}

lru = sweep[sweep.policy == "lru"].set_index("capacity")
rows = []
for capacity, row in lru.iterrows():
    entry = {
        "capacity": capacity,
        "cache_GB": row["cache_bytes"] / 1e9,
        "hit_rate": row["hit_rate"],
        "MB/token": row["fetch_bytes_per_token"] / 1e6,
    }
    for name, bandwidth in TIERS.items():
        entry[f"tok/s {name}"] = 1.0 / max(
            cachesim.seconds_per_token(row["fetch_bytes_per_token"], bandwidth), 1e-9
        )
    rows.append(entry)

display(pd.DataFrame(rows).round(2))
print("\nUpper bounds: weight movement only, ignoring compute, PCIe, and latency.")

## Where this lands

Write down the five answers before moving on — Stage 1 is designed against them:

1. **Skew** — how big does a pinned cache need to be for a useful hit rate?
2. **Locality** — recency or frequency? (watch for the LRU cliff)
3. **Predictability** — how many layers of lead time do you actually get? *(sets the whole prefetch design)*
4. **Headroom** — LRU→Belady gap: better eviction, or better prefetch?
5. **Expansion** — does batching amortise loads, or does the union scatter?

Answers 3 and 5 together decide the shape of Stage 1:

| | expansion slow | expansion fast |
|---|---|---|
| **predictable** | block-ahead prefetch, large lookahead — the best case | per-token prefetch, small lookahead |
| **unpredictable** | batch anyway; reuse carries it without prediction | neither helps — the win has to come from eviction and pinning |

The upper-left cell is where speculative decoding stops being a decode-speed
trick and becomes a *bandwidth* optimisation: one set of expert loads serving a
whole verified block. On memory-bound hardware that is the larger effect.

In [ ]:
expansion_detail = analysis.expert_set_expansion(frame, store.num_experts, store.top_k)
expansion = analysis.expansion_summary(expansion_detail, store.num_experts, store.top_k)

display(
    expansion[["block_size", "mean_unique", "expansion_ratio",
               "random_ratio", "bytes_amortization"]].round(3)
)
plots.plot_expansion(expansion, store.top_k);

In [ ]:
# Does expansion vary by depth? If late layers scatter and early ones do not,
# a block-ahead prefetcher should be depth-dependent — deep layers get a
# smaller lookahead because the union grows too fast to be worth staging.
by_depth = (
    expansion_detail[expansion_detail.block_size == 4]
    .groupby("layer")["mean_unique"]
    .mean()
    .div(store.top_k)
    .rename("expansion_at_B4")
    .reset_index()
)
display(by_depth.round(3).to_string(index=False))

## Where this lands

Write down the four answers before moving on — Stage 1 is designed against them:

1. **Skew** — how big does a pinned cache need to be for a useful hit rate?
2. **Locality** — recency or frequency?
3. **Predictability** — how many layers of lead time do you actually get? *(sets the whole prefetch design)*
4. **Headroom** — LRU→Belady gap: better eviction, or better prefetch?